In [1]:
"""
==============================================================================
USC Raw Data Processing & OBU 6-Second Validation Engine

Author      : Sanmathi S
Technology  : Python, Pandas, NumPy
Version     : Production Release

Description
-----------
This automation solution validates OBU telemetry data using forward
timestamp comparison logic and identifies transmission interval anomalies.

The system calculates the time gap between consecutive OBU records,
classifies each interval, and generates anomaly reports for analysis.

Key Features
------------
• OBU Timestamp Based Validation
• Forward Time Difference Calculation
• 6-Second Transmission Monitoring
• Automated Anomaly Detection
• Time Range Classification
• Full Dataset Export
• Anomaly-Only Report Generation
• Large Dataset Processing Support

Validation Logic
----------------
Current OBU Timestamp
            ↓
Next OBU Timestamp
            ↓
Time Difference Calculation
            ↓
Normal = 6 Seconds
Anomaly = Any Other Interval

Output Reports
--------------
1. Complete Validation Dataset
2. Anomaly-Only Report
3. Time Gap Distribution Summary

Business Benefits
-----------------
• Detects missing telemetry packets
• Identifies communication delays
• Supports fleet data quality validation
• Improves telematics monitoring accuracy
• Enables fast anomaly investigation

Domain
------
Automotive Telematics | OBU Data Analytics | Vehicle Tracking

==============================================================================
"""

import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_FILE         = r"C:\Users\HTL_Sanmathi\Downloads\USC_6Second_RawData\input\Input_MB1AUGCC6SRGK6704_2026-03-05_2026-03-08.csv"
OUTPUT_FULL        = r"C:\Users\HTL_Sanmathi\OneDrive - Ashok Leyland Ltd\Sanmathi S_Working Code\AC_Trip_Generation_2026\6_Second_Tracking_Update_Code\raw_output_MB1A5HCD5SAGU7327.csv"
OUTPUT_ANOMALIES   = r"C:\Users\HTL_Sanmathi\OneDrive - Ashok Leyland Ltd\Sanmathi S_Working Code\AC_Trip_Generation_2026\6_Second_Tracking_Update_Code\anamalies_MB1A5HCD5SAGU7327.csv"

# NORMAL definition:
STRICT_EXACT_6S   = True       # True => exactly 6.0 is normal; False => allow tolerance
TOLERANCE_SECONDS = 0.10       # used only if STRICT_EXACT_6S = False

# =============================================================================
# LOAD DATA
# =============================================================================
df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.strip()  # avoid hidden-space KeyError

# Required columns
required_cols = {"obu_id", "obu_timestamp"}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing required column(s): {missing}")

# Parse OBU timestamp (Kafka ignored for logic)
df["obu_timestamp"] = pd.to_datetime(df["obu_timestamp"], errors="coerce")
bad_ts = df["obu_timestamp"].isna().sum()
if bad_ts:
    print(f"[WARN] {bad_ts} row(s) have invalid 'obu_timestamp' (NaT). Forward diffs around them will be NaN.")

# =============================================================================
# STABLE SORT (so 'next' is deterministic)
# =============================================================================
# If S_No exists, keep it as tiebreaker to preserve input order when timestamps tie
sort_cols = [c for c in ["obu_id", "obu_timestamp", "S_No"] if c in df.columns]
if not sort_cols:
    sort_cols = ["obu_id", "obu_timestamp"]
df = df.sort_values(by=sort_cols, kind="mergesort")

# =============================================================================
# FORWARD GAP (current -> next) USING OBU ONLY
# =============================================================================
df["next_obu_timestamp"] = df.groupby("obu_id")["obu_timestamp"].shift(-1)
df["time_diff"] = (df["next_obu_timestamp"] - df["obu_timestamp"]).dt.total_seconds()

# Minutes version
df["time diff mins"] = df["time_diff"] / 60.0

# =============================================================================
# NORMAL vs ANOMALY FLAG (based on forward OBU diff)
# =============================================================================
if STRICT_EXACT_6S:
    normal_mask = df["time_diff"] == 6.0
else:
    normal_mask = np.isclose(df["time_diff"], 6.0, atol=TOLERANCE_SECONDS)

# Only rows with a next timestamp (non-NaN time_diff) can be normal/anomaly
df["is_normal"]  = df["time_diff"].notna() & normal_mask
df["is_anomaly"] = df["time_diff"].notna() & (~normal_mask)

# =============================================================================
# RANGE LABELS (based on forward OBU diff)
# =============================================================================
t = df["time_diff"]

# Protect the "Normal (6s)" bin if tolerance is enabled
if STRICT_EXACT_6S:
    normal_bin = (t == 6.0)
else:
    normal_bin = np.isclose(t, 6.0, atol=TOLERANCE_SECONDS)


conditions = [
    (t <= 0.0),
    (t <= 1.0),
    (t <= 2.0),
    (t <= 3.0),
    (t <= 4.0),
    (t > 4.0) & (t <= 5.0),
    (t > 5.0) & (t < 6.0) & (~normal_bin),
    normal_bin,
    (t > 6.0) & (~normal_bin) & (t <= 7.0),
    (t > 7.0) & (t <= 8.0),
    (t > 8.0)
]

labels = [
    "Below 0 Seconds",
    "Below 1 Seconds",
    "Below 2 Seconds",
    "Below 3 Seconds",
    "Below 4 Seconds",
    "Above 4 Seconds (within 5s)",
    "Above 5 Seconds (within 6s)",
    "Normal (6s)",
    "Above 6 Seconds (within 7s)",
    "Above 7 Seconds (within 8s)",
    "Critical Gap (Above 8s)"
]

df["time diff ranges"] = np.select(conditions, labels, default="Uncategorized")

# =============================================================================
# (OPTIONAL) REASON COLUMN — explains row inclusion/exclusion in anomalies file
# =============================================================================
def explain_row(r):
    if pd.isna(r["time_diff"]):
        return "no_next_obu_row"       # last row for this obu_id
    if r["is_normal"]:
        return "normal_6s"
    if r["is_anomaly"]:
        return "anomaly"
    return "other"

df["export_reason"] = df.apply(explain_row, axis=1)

# =============================================================================
# SAVE OUTPUTS (ALL rows + anomalies-only)
# =============================================================================
# Ensure output folders exist
Path(OUTPUT_FULL).parent.mkdir(parents=True, exist_ok=True)

# Save ALL rows with seconds in timestamps for manual verification
df.to_csv(OUTPUT_FULL, index=False, date_format="%Y-%m-%d %H:%M:%S")
print(f"📄 Full dataset (ALL rows) saved to:\n   {OUTPUT_FULL}")

# Save anomalies-only (optional convenience)
anomalies = df[df["is_anomaly"]].copy()
if not anomalies.empty:
    anomalies.to_csv(OUTPUT_ANOMALIES, index=False, date_format="%Y-%m-%d %H:%M:%S")
    print(f"🚩 Anomalies-only saved to:\n   {OUTPUT_ANOMALIES}")
else:
    print("✅ No anomalies to save (all forward OBU gaps are normal for rows with a next timestamp).")

# =============================================================================
# SUMMARY (console)
# =============================================================================
total_rows    = len(df)
with_next     = df["time_diff"].notna().sum()
normal_6s     = df["is_normal"].sum()
total_anoms   = df["is_anomaly"].sum()
no_next_rows  = df["time_diff"].isna().sum()

print("\n" + "="*80)
print("DARBY DATA SUMMARY: OBU-only, FORWARD 6-second VALIDATION (ALL ROWS)")
print("="*80)
print(f"Input file                    : {INPUT_FILE}")
print(f"Total rows                    : {total_rows}")
print(f"Rows with next OBU timestamp  : {with_next}")
print(f"  - Normal 6s                 : {normal_6s}")
print(f"  - Anomalies                 : {total_anoms}")
print(f"Rows with NO next row         : {no_next_rows} (last row per obu_id)")

# Optional: category distribution (all rows)
print("\nTIME DIFF RANGE DISTRIBUTION (all rows with a diff):")
print(df.loc[df["time_diff"].notna(), "time diff ranges"].value_counts().to_string())

print("="*80 + "\n")

📄 Full dataset (ALL rows) saved to:
   C:\Users\HTL_Sanmathi\OneDrive - Ashok Leyland Ltd\Sanmathi S_Working Code\AC_Trip_Generation_2026\6_Second_Tracking_Update_Code\raw_output_MB1A5HCD5SAGU7327.csv
🚩 Anomalies-only saved to:
   C:\Users\HTL_Sanmathi\OneDrive - Ashok Leyland Ltd\Sanmathi S_Working Code\AC_Trip_Generation_2026\6_Second_Tracking_Update_Code\anamalies_MB1A5HCD5SAGU7327.csv

DARBY DATA SUMMARY: OBU-only, FORWARD 6-second VALIDATION (ALL ROWS)
Input file                    : C:\Users\HTL_Sanmathi\Downloads\USC_6Second_RawData\input\Input_MB1AUGCC6SRGK6704_2026-03-05_2026-03-08.csv
Total rows                    : 97
Rows with next OBU timestamp  : 96
  - Normal 6s                 : 94
  - Anomalies                 : 2
Rows with NO next row         : 1 (last row per obu_id)

TIME DIFF RANGE DISTRIBUTION (all rows with a diff):
time diff ranges
Normal (6s)                    94
Above 6 Seconds (within 7s)     1
Above 4 Seconds (within 5s)     1

